In [ ]:
#!/usr/bin/env python3
"""
Simple hypothesis tests on raw hotel features.

These tests complement the permutation coefficient tests by checking whether
hotel ratings differ across interpretable groups in the raw data.

Current tests target:
  - city_population
  - attractions_count
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
from scipy.stats import pearsonr, spearmanr, ttest_ind

_ROOT = Path(__file__).resolve().parents[2]
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.modeling.feature_matrix import load_joined_hotels
from src.raw_data_paths import resolve_joined_hotels_parquet


def _welch_ttest_for_feature(df, feature: str, target: str) -> dict[str, float | str | int]:
    test_df = df.dropna(subset=[feature, target])
    median_value = test_df[feature].median()

    high_group = test_df[test_df[feature] >= median_value][target]
    low_group = test_df[test_df[feature] < median_value][target]

    t_stat, p_value = ttest_ind(high_group, low_group, equal_var=False)

    return {
        "test": "Welch two-sample t-test",
        "null_hypothesis": f"Mean {target} is equal for high vs low {feature} groups",
        "feature": feature,
        "median_split_value": float(median_value),
        "high_group_n": int(len(high_group)),
        "low_group_n": int(len(low_group)),
        "high_group_mean": float(high_group.mean()),
        "low_group_mean": float(low_group.mean()),
        "difference_in_means": float(high_group.mean() - low_group.mean()),
        "t_statistic": float(t_stat),
        "p_value": float(p_value),
    }


def _correlation_tests_for_feature(df, feature: str, target: str) -> dict[str, float | str | int]:
    test_df = df.dropna(subset=[feature, target])

    pearson_corr, pearson_p = pearsonr(test_df[feature], test_df[target])
    spearman_corr, spearman_p = spearmanr(test_df[feature], test_df[target])

    return {
        "test": "Pearson and Spearman correlation tests",
        "null_hypothesis": f"No association between {feature} and {target}",
        "feature": feature,
        "n": int(len(test_df)),
        "pearson_correlation": float(pearson_corr),
        "pearson_p_value": float(pearson_p),
        "spearman_correlation": float(spearman_corr),
        "spearman_p_value": float(spearman_p),
    }


def main() -> int:
    p = argparse.ArgumentParser(description="Simple hypothesis tests for hotel ratings.")
    p.add_argument("--joined-path", type=Path, default=None, help="Optional override path to joined Parquet")
    p.add_argument("--project-root", type=Path, default=_ROOT)
    p.add_argument("--sample", type=int, default=None, metavar="N", help="Randomly sample N rows")
    p.add_argument("--random-state", type=int, default=42)
    p.add_argument("--target", default="hotel_star_rating")
    p.add_argument(
        "--features",
        nargs="+",
        default=["city_population", "attractions_count"],
        help="Raw numeric features to test against hotel_star_rating",
    )
    p.add_argument(
        "--out",
        type=Path,
        default=_ROOT / "outputs" / "model_artifacts" / "simple_hypothesis_tests.json",
        help="Output JSON path",
    )
    args = p.parse_args()

    joined = args.joined_path or resolve_joined_hotels_parquet(args.project_root)
    df = load_joined_hotels(joined).dropna(subset=[args.target])

    if args.sample is not None:
        df = df.sample(n=min(args.sample, len(df)), random_state=args.random_state)

    results = {}

    for feature in args.features:
        if feature not in df.columns:
            print(f"Skipping {feature}: not found in dataframe")
            continue

        results[feature] = {
            "t_test": _welch_ttest_for_feature(df, feature, args.target),
            "correlation_tests": _correlation_tests_for_feature(df, feature, args.target),
        }

    out = {
        "joined_path": str(joined.resolve()),
        "rows_used": int(len(df)),
        "target": args.target,
        "features_tested": list(results.keys()),
        "tests": results,
    }

    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(out, indent=2), encoding="utf-8")

    print(f"Wrote simple hypothesis test summary to {args.out}")

    for feature, feature_results in results.items():
        t = feature_results["t_test"]
        c = feature_results["correlation_tests"]

        print(f"\n{feature}")
        print(f"  High mean: {t['high_group_mean']:.4f}")
        print(f"  Low mean:  {t['low_group_mean']:.4f}")
        print(f"  Difference: {t['difference_in_means']:.4f}")
        print(f"  Welch t-test p-value: {t['p_value']:.4f}")
        print(f"  Pearson r: {c['pearson_correlation']:.4f}, p={c['pearson_p_value']:.4f}")
        print(f"  Spearman r: {c['spearman_correlation']:.4f}, p={c['spearman_p_value']:.4f}")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())
